In [1]:
import pandas as pd

def load_and_convert(csv_file):
    # Load user data
    df = pd.read_csv(csv_file, parse_dates=["timestamp"])

    # Sort by time
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Time-based features
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek  # Monday=0, Sunday=6
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    df["minutes_since_midnight"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute

    # Track last occurrences of events
    last_drink, last_eat, last_outside = None, None, None
    drink_gaps, eat_gaps, outside_gaps = [], [], []

    for idx, row in df.iterrows():
        current_time = row["timestamp"]

        # Time since last drink
        if row["activity"] == "drink":
            if last_drink is not None:
                drink_gaps.append((current_time - last_drink).total_seconds() / 60)
            last_drink = current_time
        df.loc[idx, "time_since_last_drink"] = (
            (current_time - last_drink).total_seconds() / 60 if last_drink else None
        )

        # Time since last eat
        if row["activity"] == "eat":
            if last_eat is not None:
                eat_gaps.append((current_time - last_eat).total_seconds() / 60)
            last_eat = current_time
        df.loc[idx, "time_since_last_eat"] = (
            (current_time - last_eat).total_seconds() / 60 if last_eat else None
        )

        # Time since last outside
        if row["activity"] == "outside_in":
            last_outside = current_time
        df.loc[idx, "time_since_last_outside"] = (
            (current_time - last_outside).total_seconds() / 60 if last_outside else None
        )

    # Rolling averages
    df["avg_drink_gap"] = sum(drink_gaps) / len(drink_gaps) if drink_gaps else None
    df["avg_eat_gap"] = sum(eat_gaps) / len(eat_gaps) if eat_gaps else None

    # Target variable
    df["minutes_until_next"] = (
        df["timestamp"].shift(-1) - df["timestamp"]
    ).dt.total_seconds() / 60

    return df

if __name__ == "__main__":
    # Example usage
    input_file = "Dataset/Userdata.csv"
    features = load_and_convert(input_file)
    print(features.head(20))
    features.to_csv("Dataset/Features.csv", index=False)

    user_id     activity           timestamp  hour  day_of_week  is_weekend  \
0         1        drink 2025-09-14 01:30:00     1            6           1   
1         1          eat 2025-09-14 12:30:00    12            6           1   
2         1        drink 2025-09-14 12:30:00    12            6           1   
3         1   outside_in 2025-09-14 13:00:00    13            6           1   
4         1  outside_out 2025-09-14 13:45:00    13            6           1   
5         1        drink 2025-09-14 14:30:00    14            6           1   
6         1        drink 2025-09-14 15:15:00    15            6           1   
7         1          eat 2025-09-14 17:30:00    17            6           1   
8         1        drink 2025-09-14 17:30:00    17            6           1   
9         1        drink 2025-09-14 19:30:00    19            6           1   
10        1          eat 2025-09-14 22:30:00    22            6           1   
11        1        drink 2025-09-14 22:30:00    22  

In [4]:
import pandas as pd

# Load data and get the date
df = pd.read_csv("Dataset/Userdata.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date

# Count occurrences per day
daily = df.groupby(['date', 'activity']).size().unstack(fill_value=0).reset_index()

# Add Features (Day context)
daily['date'] = pd.to_datetime(daily['date'])
daily['day_of_week'] = daily['date'].dt.dayofweek
daily['is_weekend'] = daily['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# CREATE THE TARGET: The count of the NEXT day, shift the counts up by 1 row so "Today" has "Tomorrow's" count as a label
daily['target_drink_tomorrow'] = daily['drink'].shift(-1)
daily['target_eat_tomorrow'] = daily['eat'].shift(-1)

# Remove the last row because we don't know the future yet
daily = daily.dropna()

daily.to_csv("Dataset/Daily_Features.csv", index=False)
print("New Daily Dataset Created!")

New Daily Dataset Created!
